# 01 — Brain Tumor MRI Classification

This notebook covers the **image classification** task from the training course (Training_Course.pdf p.81-92).
We classify single-slice brain MRI into 4 categories: `glioma`, `meningioma`, `notumor`, `pituitary`.

**Three architectures are compared:**
1. `SimpleMLP` — baseline that flattens the image (lecture p.82: "spatial structure is destroyed")
2. `SimpleCNN` — the textbook 3-block CNN that matches the lecture exactly
3. `DeeperCNN` — a mini-ResNet with BatchNorm and residual connections

**Audience note (AD researchers).** This pipeline is the same one used for AD vs. CN classification on T1 MRI. The only differences in your real research code are (a) 3D volumes instead of 2D slices, (b) a subject-level train/test split to prevent leakage, and (c) clinically meaningful metrics — *all of which we demonstrate below*.

**Dataset.** Brain Tumor MRI Dataset by Masoud Nickparvar (Kaggle, downloaded via `kagglehub`).

## 0. Setup

```bash
pip install torch torchvision matplotlib numpy pillow kagglehub
```

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import sys
from pathlib import Path

# Notebook runs from cnn_handson/notebooks/. Add the project root to sys.path.
PROJ_DIR = Path.cwd().parent
sys.path.insert(0, str(PROJ_DIR))

import torch
import numpy as np
import matplotlib.pyplot as plt

from src.models import SimpleMLP, SimpleCNN, DeeperCNN, count_parameters
from src.data import (
    TUMOR_LABELS, NUM_CLASSES,
    get_device, get_classification_dataloaders,
)
from src.train import (
    train_classifier, evaluate_classifier,
    confusion_matrix, classification_report,
)
from src.viz import (
    plot_sample_grid, plot_class_distribution, plot_history,
    plot_conv_filters, plot_feature_maps,
    plot_confusion_matrix, plot_predictions, plot_gradcam,
)

torch.manual_seed(42); np.random.seed(42)
device = get_device()
print(f"Using device: {device}")

## 1. Load data

ImageFolder layout, grayscale → resize(128) → normalize. Training set gets light augmentation (horizontal flip + ±10° rotation) — safe choices for brain MRI.

In [ ]:
IMG_SIZE = 128
BATCH_SIZE = 64

train_loader, val_loader, test_loader, (train_ds, val_ds, test_ds) = get_classification_dataloaders(
    data_dir=str(PROJ_DIR / "data"),
    batch_size=BATCH_SIZE,
    img_size=IMG_SIZE,
    augment=True,
)

print(f"Train: {len(train_ds)}  Val: {len(val_ds)}  Test: {len(test_ds)}")
print(f"Classes: {TUMOR_LABELS}")

X, y = next(iter(train_loader))
print(f"Batch X: {tuple(X.shape)}  y: {tuple(y.shape)}  pixel range: [{X.min():.2f}, {X.max():.2f}]")

In [ ]:
# Sanity-check class balance — critical for medical data
_ = plot_class_distribution(train_ds.dataset, title="Train class distribution")
plt.show()
_ = plot_class_distribution(test_ds, title="Test class distribution")
plt.show()

In [ ]:
_ = plot_sample_grid(train_ds, n=16, ncols=8, title="Brain MRI samples (training set)")
plt.show()

## 2. Three models

| Model | Description | Params |
|---|---|---|
| `SimpleMLP` | Flatten + 2 FC layers — destroys spatial structure | ~4M |
| `SimpleCNN` | 3x (Conv+ReLU+MaxPool) + 2 FC — the lecture model | ~2.1M |
| `DeeperCNN` | Stem + 4 ResBlock stages + GAP + FC (mini-ResNet) | ~3M |

In [ ]:
models = {
    "mlp":       SimpleMLP(input_shape=(1, IMG_SIZE, IMG_SIZE), num_classes=NUM_CLASSES),
    "simplecnn": SimpleCNN(in_channels=1, num_classes=NUM_CLASSES, img_size=IMG_SIZE),
    "deepcnn":   DeeperCNN(in_channels=1, num_classes=NUM_CLASSES),
}
for name, m in models.items():
    print(f"{name:10s}  params: {count_parameters(m):>10,}")

In [ ]:
# Verify output-size formula from lecture p.87 directly on SimpleCNN
cnn = models["simplecnn"]
dummy = torch.randn(2, 1, IMG_SIZE, IMG_SIZE)
out = cnn.features.conv1(dummy);    print(f"after conv1: {tuple(out.shape)}")
out = cnn.features.pool1(cnn.features.relu1(out)); print(f"after pool1: {tuple(out.shape)}")
out = cnn.features.conv2(out);      print(f"after conv2: {tuple(out.shape)}")
out = cnn.features.pool2(cnn.features.relu2(out)); print(f"after pool2: {tuple(out.shape)}")
out = cnn.features.conv3(out);      print(f"after conv3: {tuple(out.shape)}")
out = cnn.features.pool3(cnn.features.relu3(out)); print(f"after pool3: {tuple(out.shape)}")
print(f"final logits: {tuple(cnn(dummy).shape)}")

## 3. Train all three models

Same hyper-parameters across models for a fair comparison: `Adam(lr=1e-3, weight_decay=1e-4)`, `CrossEntropyLoss`. The lecture training pattern (p.70-76) is encapsulated in `train_classifier`.

In [ ]:
EPOCHS = 10        # quick: 5, better: 15+
LR = 1e-3
WD = 1e-4

histories = {}
for name, model in models.items():
    print(f"\n=== Training {name} ===")
    histories[name] = train_classifier(
        model, train_loader, val_loader,
        epochs=EPOCHS, lr=LR, weight_decay=WD, device=device,
    )

In [ ]:
# Compare validation accuracy curves
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
epochs = range(1, EPOCHS + 1)
for name in models:
    ax[0].plot(epochs, histories[name]['val_loss'], 'o-', label=name)
    ax[1].plot(epochs, [a*100 for a in histories[name]['val_acc']], 'o-', label=name)
ax[0].set_xlabel('Epoch'); ax[0].set_ylabel('Val Loss'); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].set_xlabel('Epoch'); ax[1].set_ylabel('Val Accuracy (%)'); ax[1].legend(); ax[1].grid(alpha=0.3)
ax[0].set_title('Val Loss'); ax[1].set_title('Val Accuracy')
plt.tight_layout(); plt.show()

for name in models:
    print(f"{name:10s}  best val_acc = {max(histories[name]['val_acc'])*100:.2f}%")

## 4. Final test-set evaluation + clinical metrics

Sensitivity (recall) and specificity are essential whenever a misclassification has clinical cost.

In [ ]:
import torch.nn as nn

results = {}
for name, model in models.items():
    loss, acc = evaluate_classifier(model, test_loader, nn.CrossEntropyLoss(), device)
    results[name] = (loss, acc)
    print(f"{name:10s}  test_loss={loss:.4f}  test_acc={acc*100:.2f}%")

In [ ]:
# Pick the best model for the rest of the notebook
best_name = max(results, key=lambda k: results[k][1])
best_model = models[best_name]
print(f"Best model: {best_name}")

cm = confusion_matrix(best_model, test_loader, NUM_CLASSES, device)
_ = plot_confusion_matrix(cm, class_names=TUMOR_LABELS)
plt.show()

report = classification_report(cm, TUMOR_LABELS)
print(f"\n{'class':<14} {'precision':>10} {'sensitivity':>12} {'specificity':>12} {'F1':>8} {'support':>8}")
print('-' * 70)
for cls, m in report.items():
    print(f"{cls:<14} {m['precision']*100:>9.2f}% {m['sensitivity']*100:>11.2f}% "
          f"{m['specificity']*100:>11.2f}% {m['f1']*100:>7.2f}% {m['support']:>8d}")

In [ ]:
_ = plot_predictions(best_model, test_ds, n=12, ncols=6, device=device)
plt.show()

## 5. What does the CNN learn?

The next three sections only apply to `SimpleCNN` (the textbook model) because its layer-by-layer feature maps are easier to inspect.

In [ ]:
# 5-1. Learned conv-1 filters (lecture p.88)
_ = plot_conv_filters(models["simplecnn"], layer_name="conv1")
plt.show()

In [ ]:
# 5-2. Feature maps per stage (lecture p.84)
tumor_idxs = [i for i in range(len(test_ds))
              if test_ds.targets[i] != TUMOR_LABELS.index('notumor')]
idx = int(np.random.choice(tumor_idxs))
img, label = test_ds[idx]

plt.figure(figsize=(3, 3))
plt.imshow(img.squeeze().numpy() * 0.1894 + 0.1738, cmap='gray')
plt.title(f"Input: {TUMOR_LABELS[label]}"); plt.axis('off'); plt.show()

_ = plot_feature_maps(models["simplecnn"], img, device=device, max_channels=8)
plt.show()

### 5-3. Grad-CAM: where does the model look?

Accuracy alone isn't enough for clinical use. Grad-CAM highlights the input region that drove the prediction.
- Healthy sign: heatmap concentrated on the tumor.
- Suspicious sign: heatmap on skull rim or background → the model may be exploiting a spurious feature.

The same technique transfers directly to AD studies for verifying that the model attends to hippocampus / entorhinal cortex.

In [ ]:
_ = plot_gradcam(models["simplecnn"], test_ds, n=6, device=device,
                 target_layer_name="conv3")
plt.show()

## 6. Save the best checkpoint

In [ ]:
out_dir = PROJ_DIR / "outputs"
out_dir.mkdir(exist_ok=True)
torch.save({
    "model_state": best_model.state_dict(),
    "name": best_name,
    "history": histories[best_name],
    "test_acc": results[best_name][1],
    "labels": TUMOR_LABELS,
}, out_dir / f"classify_{best_name}.pt")
print("Saved.")

## 7. Hands-on exercises

### Lecture concepts to verify yourself
1. **Filter size F (p.85).** Change `kernel_size` from 3 to 5 in `SimpleCNN`. What padding `P` preserves a 128x128 output? Use the formula `O = (I + 2P − F)/S + 1`.
2. **Stride S (p.85).** Set `conv1` stride to 2 and remove the first `MaxPool2d`. Does this match the original output size? How do the parameter count and accuracy change?
3. **Dropout (p.67).** Sweep `dropout ∈ {0, 0.3, 0.6}` and compare the train-vs-val accuracy gap.
4. **Optimizer & LR (p.64-65).** Try `SGD(momentum=0.9, lr=1e-2)` vs. `Adam(lr=1e-3)`.
5. **Batch size (p.66).** Compare 16 / 64 / 256 in terms of epoch time and final accuracy.

### Medical imaging extensions
6. **Class imbalance.** Pass class weights to `CrossEntropyLoss(weight=...)`.
7. **Augmentation safety.** Disable augmentation and observe overfitting. Which augmentations are unsafe for brain MRI?
8. **Transfer learning.** Swap the backbone for `torchvision.models.resnet18(weights='IMAGENET1K_V1')` with a 1-channel first conv.
9. **Read Grad-CAM critically.** For `notumor` predictions — is the model attending to the absence of a lesion, or to other cues?
10. **Translate to AD.** What three changes would you make to apply this pipeline to ADNI (CN / MCI / AD)?